In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from transformers import AutoTokenizer
from IPython.display import display
from collections import Counter

from src.prepare_dataset import get_dpm_cats_df, get_dpm_pcl_df


/home/nik/miniforge3/envs/nlp-cw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dpm_cats_df = get_dpm_cats_df()
dpm_plc_df = get_dpm_pcl_df()

# Basic Analysis

In [3]:
print(dpm_plc_df.head())
print(dpm_plc_df.shape)
print(dpm_plc_df.columns)
print(dpm_plc_df.dtypes)

   par_id      art_id    keyword country_code  \
0       3  @@16584954  immigrant           ie   
1       4   @@7811231   disabled           nz   
2       5   @@1494111    refugee           ca   
3       6   @@9382277    in-need           in   
4       7   @@7562079    refugee           za   

                                                text  label  
0  White House press secretary Sean Spicer said t...      0  
1  Council customers only signs would be displaye...      0  
2  " Just like we received migrants fleeing El Sa...      0  
3  To bring down high blood sugar levels , insuli...      0  
4  The European Union is making an historic mista...      0  
(10467, 6)
Index(['par_id', 'art_id', 'keyword', 'country_code', 'text', 'label'], dtype='object')
par_id           int64
art_id          object
keyword         object
country_code    object
text            object
label            int64
dtype: object


In [4]:
print(f'null text rows: {dpm_plc_df["text"].isnull().sum()}')
print(f'null keyword rows: {dpm_plc_df["keyword"].isnull().sum()}')
print(f'empty text rows: {dpm_plc_df["text"].str.strip().eq("").sum()}')
print(f'empty keyword rows: {dpm_plc_df["keyword"].str.strip().eq("").sum()}')
print(f'text rows duplicates: {dpm_plc_df["text"].duplicated().sum()}')

dpm_plc_df['text'] = dpm_plc_df['text'].fillna('')

null text rows: 1
null keyword rows: 0
empty text rows: 0
empty keyword rows: 0
text rows duplicates: 0


# Label analysis

In [5]:
label_vp = dpm_plc_df['label'].value_counts(normalize=True).sort_index()
binary_label_vp = (dpm_plc_df['label'] >= 2).value_counts(normalize=True)

label_vc = dpm_plc_df['label'].value_counts().sort_index()
binary_label_vc = (dpm_plc_df['label'] >= 2).value_counts()

print(f'label value proportions/counts: \n{pd.concat([label_vp, label_vc], axis=1)} \n\nbinary label value propotions:\n{pd.concat([binary_label_vp, binary_label_vc], axis=1)}')

label value proportions/counts: 
       proportion  count
label                   
0        0.814656   8527
1        0.090475    947
2        0.013758    144
3        0.043757    458
4        0.037355    391 

binary label value propotions:
       proportion  count
label                   
False     0.90513   9474
True      0.09487    993


# Token-level EDA: RoBERTa tokenizer

In [6]:
dpm_plc_df = dpm_plc_df.copy()
dpm_plc_df['is_pcl'] = (dpm_plc_df['label'] >= 2).astype(int)

tokenizer_name = 'roberta-large'
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
MODEL_MAX_LEN = tokenizer.model_max_length 

special_ids = set(tokenizer.all_special_ids)
print('Tokenizer:', tokenizer_name)
print('Model max length (tokens):', MODEL_MAX_LEN)


Tokenizer: roberta-large
Model max length (tokens): 512


## Token-length distribution: 

In [7]:
def batched(iterable, batch_size):
    for i in range(0, len(iterable), batch_size):
        yield iterable[i:i+batch_size]

def compute_token_lengths(texts, batch_size=256):
    lengths = []
    texts = texts
    for batch in batched(texts, batch_size):
        enc = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
            return_attention_mask=False,
            return_token_type_ids=False,
        )
        lengths.extend(enc['length'])
    return np.array(lengths, dtype=np.int32)

texts = dpm_plc_df['text'].fillna('').astype(str).tolist()
lengths = compute_token_lengths(texts)
dpm_plc_df['n_tokens_roberta'] = lengths

ps = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print(dpm_plc_df['n_tokens_roberta'].describe(ps))

fig = px.histogram(
    dpm_plc_df,
    x='n_tokens_roberta',
    color='is_pcl',
    nbins=1000,
    barmode='overlay',
    title='tok length dist',
    histnorm='probability'
)
fig.add_vline(
    x=MODEL_MAX_LEN,
    line_width=2,
    line_dash='dash',
)
fig.update_layout(xaxis_title='num tokens', yaxis_title='prob')
fig.show()

def truncation_table(lengths, trunc_lengths):
    rows = []
    for m in trunc_lengths:
        trunc_rate = float(np.mean(lengths > m))
        kept_frac = float(np.mean(np.minimum(lengths, m) / lengths))
        rows.append({'max_length': m, 'pct_truncated': 100*trunc_rate, 'avg_fraction_tokens_kept': kept_frac})
    return pd.DataFrame(rows)

trunc_df = truncation_table(lengths, trunc_lengths=[64, 96, 128, 192, 256, 384, MODEL_MAX_LEN])
trunc_df


Token indices sequence length is longer than the specified maximum sequence length for this model (545 > 512). Running this sequence through the model will result in indexing errors


count    10467.000000
mean        55.482373
std         32.267273
min          2.000000
1%          12.000000
5%          20.000000
10%         25.000000
25%         35.000000
50%         48.000000
75%         68.000000
90%         94.000000
95%        114.000000
99%        159.000000
max       1004.000000
Name: n_tokens_roberta, dtype: float64


,max_length,pct_truncated,avg_fraction_tokens_kept
0,64,28.011847,0.925163
1,96,9.095252,0.981338
2,128,3.057227,0.995024
3,192,0.257954,0.999447
4,256,0.076431,0.999741
5,384,0.038215,0.999891
6,512,0.019108,0.999947


## Class token association

In [11]:
def iter_text_token_ids(texts, max_length=MODEL_MAX_LEN):
    for text in texts:
        enc = tokenizer(
            text,
            add_special_tokens=False,
            truncation=True,
            max_length=max_length,
            padding=False,
        )
        yield enc['input_ids']

pos_texts = (
    dpm_plc_df.loc[dpm_plc_df['is_pcl'] == 1, 'text']
    .fillna('')
    .astype(str)
    .tolist()
)
neg_texts = (
    dpm_plc_df.loc[dpm_plc_df['is_pcl'] == 0, 'text']
    .fillna('')
    .astype(str)
    .tolist()
)

pos_mass_counts = Counter()
pos_unique_counts = Counter()
neg_mass_counts = Counter()
neg_unique_counts = Counter()

for ids in iter_text_token_ids(pos_texts):
    pos_mass_counts.update(ids)
    pos_unique_counts.update(set(ids))

for ids in iter_text_token_ids(neg_texts):
    neg_mass_counts.update(ids)
    neg_unique_counts.update(set(ids))

total_pos_mass = sum(pos_mass_counts.values())
total_neg_mass = sum(neg_mass_counts.values())
voacb_size = len(set(pos_mass_counts.keys()) | set(neg_mass_counts))

def log_odds(tid: int, alpha=0.1) -> float:
    cp = pos_mass_counts.get(tid, 0)
    cn = neg_mass_counts.get(tid, 0)
    lp = np.log((cp + alpha) / (total_pos_mass + alpha * voacb_size))
    ln = np.log((cn + alpha) / (total_neg_mass + alpha * voacb_size))
    return float(lp - ln)

rows = []
all_tids = set(pos_mass_counts) | set(neg_mass_counts)

for tid in all_tids:
    tok = tokenizer.convert_ids_to_tokens(int(tid))
    rows.append({
        'token': tok.replace('Ġ', ''),
        'count_pos': pos_mass_counts.get(tid, 0),
        'count_neg': neg_mass_counts.get(tid, 0),
        'log_odds_pos_minus_neg': log_odds(int(tid)),
        'unique_count_pos': pos_unique_counts.get(tid, 0),
        'unique_count_neg': neg_unique_counts.get(tid, 0),
    })

assoc_df = pd.DataFrame(rows)

min_count = 20
assoc_df = assoc_df[(assoc_df['count_pos'] + assoc_df['count_neg']) >= min_count]

top_pos = assoc_df.sort_values('log_odds_pos_minus_neg', ascending=False).head(25)
top_neg = assoc_df.sort_values('log_odds_pos_minus_neg', ascending=True).head(25)

print(f'vocab size: {voacb_size}, Roberta voacb size: {tokenizer.vocab_size}')
print('top PCL tokens by positive log-odds:')
display(top_pos)

print('top non-PCL tokens by negative log-odds:')
display(top_neg)

vocab size: 28340, Roberta voacb size: 50265
top PCL tokens by positive log-odds:


,token,count_pos,count_neg,log_odds_pos_minus_neg,unique_count_pos,unique_count_neg
10445,hungry,23,10,2.943460,23,10
7928,donate,16,7,2.934888,16,7
1567,Christmas,28,14,2.805758,25,14
21948,needy,13,8,2.596911,12,8
20936,itute,12,9,2.401094,10,9
11212,dignity,13,11,2.281830,13,11
13863,dest,12,11,2.202423,10,11
1784,God,34,34,2.116163,29,30
3896,feed,18,20,2.011355,15,19
4231,Every,10,12,1.935493,8,12


top non-PCL tokens by negative log-odds:


,token,count_pos,count_neg,log_odds_pos_minus_neg,unique_count_pos,unique_count_neg
1430,anti,0,152,-5.210960,0,123
1074,Chinese,0,91,-4.698380,0,66
314,points,0,72,-4.464476,0,66
677,released,0,65,-4.362347,0,63
230,3,0,58,-4.248588,0,54
1727,Act,0,56,-4.213558,0,53
1129,issued,0,53,-4.158599,0,50
470,News,0,53,-4.158599,0,50
1081,compared,0,52,-4.139587,0,49
760,Bank,0,47,-4.038695,0,38
